# Real-world data coding for neuroscientists (ReCoN)

### MSc in Translational Neuroscience,

### Department of Brain Sciences, Faculty of Medicine,

### Imperial College London

### Autumn 2026

# 2. Introduction to tabular data with pandas
#### Working with Electronic Health Records (EHR) from people with Parkinson's disease

### Introduction to tabular data in healthcare

---

Much of the data you will work with in clinical research is **tabular**: it is organised in rows and columns, like an Excel spreadsheet. **Electronic health records (EHRs)**, clinical registries and clinical trial databases are all stored this way:

- each **row** is usually one **observation** — for example, one patient's visit to a clinic;
- each **column** is one **attribute** of that observation — for example, the patient ID, the visit date or the medication dose.

In the last session you stored numbers in **NumPy arrays**. Real clinical records, however, mix numbers, text and dates, and every column has a name. **pandas** is the Python library designed for exactly this kind of table.

---

#### **What you will learn today**

- Create a table (a **DataFrame**), load one from a file and save it
- Look inside a table to understand what it contains
- Select columns and rows, and filter rows using conditions
- Add, rename and remove columns
- Fix data types, deal with missing values and clean messy text
- Summarise rows by group (e.g. one result per patient) and combine tables

---

#### **How this notebook works**

Every topic follows the same three steps:

1.  **Concept** — a short explanation of the idea
2.  **Example** — a worked example. Read it, run it, and make sure you understand the output
3.  **Task** — a small exercise on a different example of the same idea. Write your code inside the task cell

> 💡 Run the cells **in order, from top to bottom** (`Shift + Enter`). Many cells use variables created in earlier cells. If something stops working, restart the kernel and run all the cells above the one you are working on.

📚 **Supporting material:** the [pandas cheat sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) is a one-page reference worth keeping open during the session.

![pandas meme](https://i.programmerhumor.io/2025/03/bf92b0031ba27033fcf1534cd911e834c963afbc0c31b90fc9550eef0fafed92.png)

In [ ]:
# pandas is usually imported with the nickname "pd", so we can write pd.something instead of pandas.something
import pandas as pd

## 2.1 Getting started with DataFrames

### 2.1.1 Creating a DataFrame

<div style="background-color:transparent; border-left:5px solid #5DADE2; padding:10px; margin-bottom:15px;">

 **Concept**

A **DataFrame** is pandas' name for a table. Every column has a **name**, and every row has a **label** called the **index**. By default the index is `0, 1, 2, ...`, just like the positions in a NumPy array.

One easy way to create a DataFrame is from a **dictionary**. A dictionary stores `key: value` pairs inside curly brackets `{}`. To build a table:

- each **key** becomes a **column name**;
- each **value** is a **list** with the contents of that column (one item per row).

```python
my_table = pd.DataFrame({
    "column_1": [row_0_value, row_1_value, row_2_value],
    "column_2": [row_0_value, row_1_value, row_2_value],
})
```

⚠️ All the lists must have the **same length**: every row needs a value in every column.

💡 Each column of a DataFrame behaves a lot like a NumPy array from the last session, but with a name attached to it.

</div>

In [ ]:
#  Example: a small table with three patients from a Parkinson's disease study
patients = pd.DataFrame({
    "patient_id": ["P01", "P02", "P03"],
    "age_at_diagnosis": [66, 55, 50],
    "smoking_status": ["Former", "Never", "Never"],
})

# Writing the name of a variable on the last line of a cell displays it
patients

,patient_id,age_at_diagnosis,smoking_status
0,P01,66,Former
1,P02,55,Never
2,P03,50,Never


In [ ]:
#  TASK 2.1.1
# Create a DataFrame called `medications` that describes three Parkinson's disease medications.
# It should have three columns: drug_name, "drug_class", and "route". 
#    "drug_name":  "Levodopa", "Rotigotine", "Pramipexole"
#    "drug_class": "Dopamine precursor", "Dopamine agonist", "Dopamine agonist"
#    "route":      "Oral", "Skin patch", "Oral"
# Then display the table.

# Write your code here:

Before the next topic, run the cell below. It creates the data file we will use for the rest of the session.

In [ ]:
# NOTE: don't worry about this cell.
# In a real project somebody would give you this file. Here we create it ourselves so that the notebook works anywhere.
# (You can already recognise what it does: it builds a DataFrame from a dictionary and saves it as a CSV file.)

pd.DataFrame({
    "patient_id": ["P01", "P02", "P03", "P04", "P05", "P01", "P02", "P03", "P05", "P06", "P01", "P02", "P05", "P04"],
    "sex": ["Male", "Female", "Female", "Male", "Male", "Male", "Female", "Female", "Male", "Female", "Male", "Female", "Male", "Male"],
    "date_of_birth": ["1950-05-14", "1965-08-22", "1970-11-30", "1980-01-15", "1958-03-02", "1950-05-14", "1965-08-22",
                      "1970-11-30", "1958-03-02", "1955-12-01", "1950-05-14", "1965-08-22", "1958-03-02", "1980-01-15"],
    "visit_date": ["2023-01-10", "2023-01-15", "2023-02-01", "2023-03-14", "2023-04-21", "2023-07-12", "2023-09-03",
                   "2023-10-06", "2023-11-30", "2024-01-09", "2024-02-08", "2024-03-20", "2024-05-17", "2024-06-04"],
    "clinic_id": [101, 102, 101, 103, 102, 101, 102, 101, 102, 103, 101, 102, 102, 103],
    "updrs_iii": [22, 30, 40, 12, 35, 25, None, 42, 38, 27, 28, 31, 41, None],
    "ledd_mg": [300, 150, 800, 0, 450, 400, 300, 850, 600, 400, 500, 300, 750, 0],
    "medication": ["Co-careldopa", "Rotigotine", "Co-beneldopa", None, "Pramipexole", "co-careldopa ", "rotigotine",
                   "co-beneldopa", "Co-careldopa + Entacapone", "Ropinirole", "CO-CARELDOPA", " Rotigotine",
                   "co-careldopa + entacapone ", None],
    "bmi": ["22.5", "25.0", "30.0", "28.0", "26.4", "22.7", "not recorded", "31.2", "26.1", "21.9", "23.1", "24.0",
            "not recorded", "27.6"],
}).to_csv("data/ehr_parkinsons.csv", index=False)

print("File 'ehr_parkinsons.csv' created!")

File 'ehr_parkinsons.csv' created!


### 2.1.2 Loading and saving data

<div style="background-color:transparent; border-left:5px solid #48C9B0; padding:10px; margin-bottom:15px;">

 **Concept**

In real projects you rarely type data in by hand: you receive it as a **file**. The most common format for tables is **CSV** (*comma-separated values*): a plain-text file where each line is a row and the values are separated by commas. You can also open CSV files in Excel.

| What you want | Code |
|:--|:--|
| **Load** a CSV file into a DataFrame | `df = pd.read_csv("file_name.csv")` |
| **Save** a DataFrame as a CSV file | `df.to_csv("file_name.csv", index=False)` |

- `pd.read_csv()` gives you back a DataFrame, so store it in a variable. By convention, the main table is often called `df`.
- `index=False` tells pandas **not** to save the row labels (`0, 1, 2, ...`) as an extra column.
- If the file is in the same folder as your notebook, its name is enough. Otherwise, write the full path to the file.

</div>

In [ ]:
#  Example 1: load the EHR file into a DataFrame called df
df = pd.read_csv("data/ehr_parkinsons.csv")
df

,patient_id,sex,date_of_birth,visit_date,clinic_id,updrs_iii,ledd_mg,medication,bmi
0,P01,Male,1950-05-14,2023-01-10,101,22.0,300,Co-careldopa,22.5
1,P02,Female,1965-08-22,2023-01-15,102,30.0,150,Rotigotine,25.0
2,P03,Female,1970-11-30,2023-02-01,101,40.0,800,Co-beneldopa,30.0
3,P04,Male,1980-01-15,2023-03-14,103,12.0,0,NaN,28.0
4,P05,Male,1958-03-02,2023-04-21,102,35.0,450,Pramipexole,26.4
5,P01,Male,1950-05-14,2023-07-12,101,25.0,400,co-careldopa,22.7
6,P02,Female,1965-08-22,2023-09-03,102,NaN,300,rotigotine,not recorded
7,P03,Female,1970-11-30,2023-10-06,101,42.0,850,co-beneldopa,31.2
8,P05,Male,1958-03-02,2023-11-30,102,38.0,600,Co-careldopa + Entacapone,26.1
9,P06,Female,1955-12-01,2024-01-09,103,27.0,400,Ropinirole,21.9


In [ ]:
#  Example 2: save the small `patients` table from 2.1.1 as a CSV file...
patients.to_csv("data/patients.csv", index=False)

# ...and load it back to check that it worked
patients_from_file = pd.read_csv("data/patients.csv")
patients_from_file

,patient_id,age_at_diagnosis,smoking_status
0,P01,66,Former
1,P02,55,Never
2,P03,50,Never


<div style="background-color:transparent; border-left:5px solid #5DADE2; padding:15px; border-radius:8px;">

<h3 style="color:#2C3E50;">Our dataset: a (synthetic) Parkinson's disease EHR extract</h3>

Each **row** is **one clinic visit**. Patients attend the clinic several times, so most patients appear in more than one row.

| **Column** | **Description** |
|:--|:--|
| **`patient_id`** | Unique identifier of each patient (P01 to P06). |
| **`sex`** | Sex recorded in the health record. |
| **`date_of_birth`** | Date of birth (YYYY-MM-DD). |
| **`visit_date`** | Date of the clinic visit (YYYY-MM-DD). |
| **`clinic_id`** | Identifier of the clinic where the visit took place. |
| **`updrs_iii`** | **MDS-UPDRS Part III** score: a clinician-rated assessment of motor symptoms in Parkinson's disease. It ranges from 0 to 132; higher means more severe. |
| **`ledd_mg`** | **Levodopa equivalent daily dose (LEDD)** in mg: the total daily dose of Parkinson's medication, converted to a common scale so that different drugs can be compared. 0 means no medication. |
| **`medication`** | Name of the prescribed medication, as typed by the clinician. |
| **`bmi`** | Body mass index (kg/m²). |

The data are made up, but they contain the same kinds of problems you will find in real EHRs: **missing values**, **numbers stored as text** and **inconsistently typed medication names**. We will find and fix them step by step.

</div>

In [ ]:
#  TASK 2.1.2
# 1. Save the `medications` table you created in Task 2.1.1 as a file called "medications.csv" (without the index).
# 2. Load the file back into a new variable called `medications_from_file` and display it.

# Write your code here:

### 2.1.3 Inspecting a DataFrame

<div style="background-color:transparent; border-left:5px solid #F4D03F; padding:10px; margin-bottom:15px;">

 **Concept**

The first thing to do with any new dataset is to **look at it**. These are the most useful tools:

| Code | What it shows |
|:--|:--|
| `df.head()` | The first 5 rows (`df.head(10)` shows the first 10) |
| `df.tail()` | The last 5 rows |
| `df.shape` | The size of the table: `(number of rows, number of columns)` |
| `df.columns` | The column names |
| `df.info()` | A summary: column names, data types and **how many non-missing values** each column has |

Notice that `shape` and `columns` have **no brackets**: they are pieces of information stored in the DataFrame (you used `array.shape` with NumPy in the same way). `head()`, `tail()` and `info()` have brackets because they are **methods**: functions that belong to the DataFrame and can take inputs, like the `10` in `df.head(10)`.

💡 Jupyter only displays the result of the **last line** of a cell. To show several results from the same cell, use `print()` for simple values or `display()` for tables.

</div>

In [ ]:
#  Example 1: the first 5 rows
df.head()

,patient_id,sex,date_of_birth,visit_date,clinic_id,updrs_iii,ledd_mg,medication,bmi
0,P01,Male,1950-05-14,2023-01-10,101,22.0,300,Co-careldopa,22.5
1,P02,Female,1965-08-22,2023-01-15,102,30.0,150,Rotigotine,25.0
2,P03,Female,1970-11-30,2023-02-01,101,40.0,800,Co-beneldopa,30.0
3,P04,Male,1980-01-15,2023-03-14,103,12.0,0,NaN,28.0
4,P05,Male,1958-03-02,2023-04-21,102,35.0,450,Pramipexole,26.4


In [ ]:
#  Example 2: size of the table and column names
print("Shape (rows, columns):", df.shape)
print("Number of visits (rows):", df.shape[0])
print("Number of columns:", df.shape[1])
print("Column names:", list(df.columns))

Shape (rows, columns): (14, 9)
Number of visits (rows): 14
Number of columns: 9
Column names: ['patient_id', 'sex', 'date_of_birth', 'visit_date', 'clinic_id', 'updrs_iii', 'ledd_mg', 'medication', 'bmi']


In [ ]:
#  TASK 2.1.3
# 1. Display the LAST 3 rows of df.
# 2. Use df.info() to find out which columns contain missing values.
#    HINT: compare each column's "Non-Null Count" with the total number of rows.

# Write your code here:

## 2.2 Selecting and filtering data

### 2.2.1 Selecting columns

<div style="background-color:transparent; border-left:5px solid #85929E; padding:10px; margin-bottom:15px;">

 **Concept**

To work with only some of the columns, write their names inside square brackets:

| Code | Result |
|:--|:--|
| `df["column_name"]` | **One** column. The result is a **Series**: a single column with its index |
| `df[["column_1", "column_2"]]` | **Several** columns. The result is a smaller **DataFrame** |

Notice the **double square brackets** when selecting several columns: the outer brackets mean *"select from df"*, and the inner brackets are a Python **list** of column names.

⚠️ Column names must be written **exactly** as they appear, including capital letters: `df["UPDRS_III"]` gives a `KeyError` because the column is called `"updrs_iii"`.

</div>

In [ ]:
#  Example 1: one column (a Series)
df["patient_id"]

0     P01
1     P02
2     P03
3     P04
4     P05
5     P01
6     P02
7     P03
8     P05
9     P06
10    P01
11    P02
12    P05
13    P04
Name: patient_id, dtype: str

In [ ]:
#  Example 2: several columns (a DataFrame). Note the double brackets!
df[["patient_id", "visit_date", "updrs_iii"]]

,patient_id,visit_date,updrs_iii
0,P01,2023-01-10,22.0
1,P02,2023-01-15,30.0
2,P03,2023-02-01,40.0
3,P04,2023-03-14,12.0
4,P05,2023-04-21,35.0
5,P01,2023-07-12,25.0
6,P02,2023-09-03,NaN
7,P03,2023-10-06,42.0
8,P05,2023-11-30,38.0
9,P06,2024-01-09,27.0


In [ ]:
#  TASK 2.2.1
# 1. Select only the "ledd_mg" column.
# 2. Select the "patient_id", "medication" and "ledd_mg" columns together, and show only the first 3 rows.
#    HINT: you can add .head(3) at the end of your selection.

# Write your code here:

### 2.2.2 Selecting rows by position (`.iloc`) and by label (`.loc`)

<div style="background-color:transparent; border-left:5px solid #5DADE2; padding:10px; margin-bottom:15px;">

 **Concept**

There are two ways to pick specific rows (and columns):

**`.iloc[ ]`: by position** (*i*nteger *loc*ation). It counts from 0 and works exactly like indexing and slicing a NumPy array:

| Code | Result |
|:--|:--|
| `df.iloc[0]` | The first row |
| `df.iloc[-1]` | The last row |
| `df.iloc[2:5]` | The rows at positions 2, 3 and 4 (the end position is **not** included) |
| `df.iloc[0:3, 0:2]` | The first 3 rows and the first 2 columns |

**`.loc[ ]`: by label**, using the row label (the index) and the column **name**:

| Code | Result |
|:--|:--|
| `df.loc[4, "medication"]` | The value in the row labelled 4, column `"medication"` |
| `df.loc[4, ["patient_id", "medication"]]` | Two columns from the row labelled 4 |

💡 At the moment the row labels are `0, 1, 2, ...`, so positions and labels match. They stop matching after you sort or filter a table (you will see this in 2.2.5).

</div>

In [ ]:
#  Example 1: the first row (position 0)
df.iloc[0]

patient_id                P01
sex                      Male
date_of_birth      1950-05-14
visit_date         2023-01-10
clinic_id                 101
updrs_iii                22.0
ledd_mg                   300
medication       Co-careldopa
bmi                      22.5
Name: 0, dtype: object

In [ ]:
#  Example 2: the rows at positions 0, 1 and 2, and the first 4 columns
df.iloc[0:3, 0:4]

,patient_id,sex,date_of_birth,visit_date
0,P01,Male,1950-05-14,2023-01-10
1,P02,Female,1965-08-22,2023-01-15
2,P03,Female,1970-11-30,2023-02-01


In [ ]:
#  Example 3: the medication recorded in the row labelled 4
df.loc[4, "medication"]

'Pramipexole'

In [ ]:
#  TASK 2.2.2
# 1. Use .iloc to select the LAST 3 rows of df.
#    HINT: negative positions count from the end, like in NumPy.
# 2. Use .loc to get the UPDRS III score ("updrs_iii") recorded in the row labelled 5.

# Write your code here:

### 2.2.3 Filtering rows with a condition

<div style="background-color:transparent; border-left:5px solid #48C9B0; padding:10px; margin-bottom:15px;">

 **Concept**

Most of the time you want the rows that meet a **condition**: *"all the visits of patient P01"*, or *"all the visits with a UPDRS III score above 30"*. Filtering works in two steps, just like boolean masks in NumPy:

1. Write a **condition** on a column. pandas checks it for every row and gives back one `True` or `False` per row. This is called a **mask**.
2. Put the mask inside `df[ ]`. Only the rows where the mask is `True` are kept.

```python
df[df["column_name"] > value]
```

| Operator | Meaning |
|:--|:--|
| `==` | equal to (two `=` signs!) |
| `!=` | not equal to |
| `>`, `<` | greater than, less than |
| `>=`, `<=` | greater than or equal to, less than or equal to |

⚠️ A single `=` **stores** a value in a variable; a double `==` **compares** two values.

⚠️ Text values need quotes (`df["sex"] == "Female"`), numbers do not (`df["clinic_id"] == 101`).

</div>

In [ ]:
#  Example 1 (step 1): the condition gives one True/False per row. This is the mask
df["updrs_iii"] > 30

0     False
1     False
2      True
3     False
4      True
5     False
6     False
7      True
8      True
9     False
10    False
11     True
12     True
13    False
Name: updrs_iii, dtype: bool

In [ ]:
#  Example 1 (step 2): use the mask to keep only the rows where it is True
# (rows with a missing score give False, so they are not kept)
df[df["updrs_iii"] > 30]

,patient_id,sex,date_of_birth,visit_date,clinic_id,updrs_iii,ledd_mg,medication,bmi
2,P03,Female,1970-11-30,2023-02-01,101,40.0,800,Co-beneldopa,30.0
4,P05,Male,1958-03-02,2023-04-21,102,35.0,450,Pramipexole,26.4
7,P03,Female,1970-11-30,2023-10-06,101,42.0,850,co-beneldopa,31.2
8,P05,Male,1958-03-02,2023-11-30,102,38.0,600,Co-careldopa + Entacapone,26.1
11,P02,Female,1965-08-22,2024-03-20,102,31.0,300,Rotigotine,24.0
12,P05,Male,1958-03-02,2024-05-17,102,41.0,750,co-careldopa + entacapone,not recorded


In [ ]:
#  Example 2: the same idea works with text. All the visits of patient P01:
df[df["patient_id"] == "P01"]

,patient_id,sex,date_of_birth,visit_date,clinic_id,updrs_iii,ledd_mg,medication,bmi
0,P01,Male,1950-05-14,2023-01-10,101,22.0,300,Co-careldopa,22.5
5,P01,Male,1950-05-14,2023-07-12,101,25.0,400,co-careldopa,22.7
10,P01,Male,1950-05-14,2024-02-08,101,28.0,500,CO-CARELDOPA,23.1


In [ ]:
#  TASK 2.2.3
# 1. Select all the visits of female patients.
# 2. Select all the visits where the patient was NOT taking any medication (ledd_mg equal to 0).
# 3. How many visits did you find in question 2?
#    HINT: len() or .shape

# Write your code here:

### 2.2.4 Combining conditions

<div style="background-color:transparent; border-left:5px solid #F4D03F; padding:10px; margin-bottom:15px;">

 **Concept**

You can combine several conditions into one mask:

| Operator | Meaning | Keeps a row when... |
|:--|:--|:--|
| `&` | AND | **both** conditions are `True` |
| `\|` | OR | **at least one** condition is `True` |
| `~` | NOT | the condition is `False` (it flips `True` and `False`) |

```python
df[(df["sex"] == "Male") & (df["ledd_mg"] > 300)]
```

⚠️ Put **each condition inside its own round brackets**. Without them, Python combines things in the wrong order and gives an error.

To filter rows **and** choose columns at the same time, use `.loc[rows, columns]`:

```python
df.loc[condition, ["column_1", "column_2"]]
```

💡 You will use `&` a lot in the next session, to pick one subject, one wrist and one task from accelerometer data.

</div>

In [ ]:
#  Example 1: visits of male patients AND with a dose above 300 mg
df[(df["sex"] == "Male") & (df["ledd_mg"] > 300)]

,patient_id,sex,date_of_birth,visit_date,clinic_id,updrs_iii,ledd_mg,medication,bmi
4,P05,Male,1958-03-02,2023-04-21,102,35.0,450,Pramipexole,26.4
5,P01,Male,1950-05-14,2023-07-12,101,25.0,400,co-careldopa,22.7
8,P05,Male,1958-03-02,2023-11-30,102,38.0,600,Co-careldopa + Entacapone,26.1
10,P01,Male,1950-05-14,2024-02-08,101,28.0,500,CO-CARELDOPA,23.1
12,P05,Male,1958-03-02,2024-05-17,102,41.0,750,co-careldopa + entacapone,not recorded


In [ ]:
#  Example 2: visits of patient P01 OR patient P03, showing only three columns
df.loc[(df["patient_id"] == "P01") | (df["patient_id"] == "P03"), ["patient_id", "visit_date", "ledd_mg"]]

,patient_id,visit_date,ledd_mg
0,P01,2023-01-10,300
2,P03,2023-02-01,800
5,P01,2023-07-12,400
7,P03,2023-10-06,850
10,P01,2024-02-08,500


In [ ]:
#  TASK 2.2.4
# 1. Select the visits of female patients with a UPDRS III score above 30.
# 2. Select the visits that took place at clinic 101 OR clinic 103,
#    and show only the "patient_id", "clinic_id" and "visit_date" columns.

# Write your code here:

### 2.2.5 Sorting rows

<div style="background-color:transparent; border-left:5px solid #85929E; padding:10px; margin-bottom:15px;">

 **Concept**

`.sort_values()` reorders the rows of a table:

| Code | Result |
|:--|:--|
| `df.sort_values("column")` | Smallest to largest (A to Z for text, oldest to newest for dates) |
| `df.sort_values("column", ascending=False)` | Largest to smallest |
| `df.sort_values(["column_1", "column_2"])` | Sort by `column_1`; rows with the same value are then sorted by `column_2` |

Sorting by patient and then by date is the usual way to read **longitudinal** records: each patient's visits appear together, in the order they happened.

⚠️ Most pandas methods **do not change** `df`. They give you back a **new** table. To keep the result, store it in a variable, for example `df_sorted = df.sort_values("visit_date")`.

</div>

In [ ]:
#  Example 1: highest dose first
df.sort_values("ledd_mg", ascending=False)

,patient_id,sex,date_of_birth,visit_date,clinic_id,updrs_iii,ledd_mg,medication,bmi
7,P03,Female,1970-11-30,2023-10-06,101,42.0,850,co-beneldopa,31.2
2,P03,Female,1970-11-30,2023-02-01,101,40.0,800,Co-beneldopa,30.0
12,P05,Male,1958-03-02,2024-05-17,102,41.0,750,co-careldopa + entacapone,not recorded
8,P05,Male,1958-03-02,2023-11-30,102,38.0,600,Co-careldopa + Entacapone,26.1
10,P01,Male,1950-05-14,2024-02-08,101,28.0,500,CO-CARELDOPA,23.1
4,P05,Male,1958-03-02,2023-04-21,102,35.0,450,Pramipexole,26.4
5,P01,Male,1950-05-14,2023-07-12,101,25.0,400,co-careldopa,22.7
9,P06,Female,1955-12-01,2024-01-09,103,27.0,400,Ropinirole,21.9
0,P01,Male,1950-05-14,2023-01-10,101,22.0,300,Co-careldopa,22.5
6,P02,Female,1965-08-22,2023-09-03,102,NaN,300,rotigotine,not recorded


In [ ]:
#  Example 2: each patient's visits together, in date order
df_sorted = df.sort_values(["patient_id", "visit_date"])
df_sorted

# Look at the row labels on the left: each row kept its original label.
# Now df_sorted.iloc[1] (second POSITION) and df_sorted.loc[1] (row LABELLED 1) are different rows!

,patient_id,sex,date_of_birth,visit_date,clinic_id,updrs_iii,ledd_mg,medication,bmi
0,P01,Male,1950-05-14,2023-01-10,101,22.0,300,Co-careldopa,22.5
5,P01,Male,1950-05-14,2023-07-12,101,25.0,400,co-careldopa,22.7
10,P01,Male,1950-05-14,2024-02-08,101,28.0,500,CO-CARELDOPA,23.1
1,P02,Female,1965-08-22,2023-01-15,102,30.0,150,Rotigotine,25.0
6,P02,Female,1965-08-22,2023-09-03,102,NaN,300,rotigotine,not recorded
11,P02,Female,1965-08-22,2024-03-20,102,31.0,300,Rotigotine,24.0
2,P03,Female,1970-11-30,2023-02-01,101,40.0,800,Co-beneldopa,30.0
7,P03,Female,1970-11-30,2023-10-06,101,42.0,850,co-beneldopa,31.2
3,P04,Male,1980-01-15,2023-03-14,103,12.0,0,NaN,28.0
13,P04,Male,1980-01-15,2024-06-04,103,NaN,0,NaN,27.6


In [ ]:
#  TASK 2.2.5
# 1. Sort df so that the visits with the highest UPDRS III scores come first, and show only the top 3 rows.
# 2. Find the 5 most recent visits: sort by "visit_date" from newest to oldest and show the first 5 rows.

# Write your code here:

## 2.3 Changing and cleaning data

### 2.3.1 Adding, renaming and removing columns

<div style="background-color:transparent; border-left:5px solid #5DADE2; padding:10px; margin-bottom:15px;">

 **Concept**

| What you want | Code |
|:--|:--|
| **Add** a new column | `df["new_column"] = ...` |
| **Rename** columns | `df = df.rename(columns={"old_name": "new_name"})` |
| **Remove** columns | `df = df.drop(columns=["column_1", "column_2"])` |

A new column is usually calculated from existing ones. Just like with NumPy arrays, the calculation is applied to **every row at once**:

```python
df["ledd_g"] = df["ledd_mg"] / 1000        # arithmetic: +, -, *, /
df["on_medication"] = df["ledd_mg"] > 0    # a condition gives a True/False column
```

Remember from 2.2.5: `rename()` and `drop()` give you back a **new** table, so write `df = ...` to keep the change.

</div>

In [ ]:
#  Example 1: two new columns
# Convert the dose from milligrams to grams (calculated for every row at once)
df["ledd_g"] = df["ledd_mg"] / 1000

# A True/False column: was the patient taking any medication at this visit?
df["on_medication"] = df["ledd_mg"] > 0

df[["patient_id", "ledd_mg", "ledd_g", "on_medication"]]

,patient_id,ledd_mg,ledd_g,on_medication
0,P01,300,0.30,True
1,P02,150,0.15,True
2,P03,800,0.80,True
3,P04,0,0.00,False
4,P05,450,0.45,True
5,P01,400,0.40,True
6,P02,300,0.30,True
7,P03,850,0.85,True
8,P05,600,0.60,True
9,P06,400,0.40,True


In [ ]:
#  Example 2: rename a column. The dictionary means {"old name": "new name"}
df = df.rename(columns={"ledd_g": "ledd_grams"})
print(list(df.columns))

#  Example 3: remove the two columns we just created (we will not need them again)
df = df.drop(columns=["ledd_grams", "on_medication"])
print(list(df.columns))

['patient_id', 'sex', 'date_of_birth', 'visit_date', 'clinic_id', 'updrs_iii', 'ledd_mg', 'medication', 'bmi', 'ledd_grams', 'on_medication']
['patient_id', 'sex', 'date_of_birth', 'visit_date', 'clinic_id', 'updrs_iii', 'ledd_mg', 'medication', 'bmi']


In [ ]:
#  TASK 2.3.1
# The MDS-UPDRS Part III score goes from 0 to 132.
# 1. Add a column "updrs_iii_percent" with the score as a percentage of the maximum: updrs_iii / 132 * 100
# 2. Add a True/False column "high_dose" that is True when "ledd_mg" is above 500.
# 3. Display the "patient_id", "updrs_iii", "updrs_iii_percent" and "high_dose" columns.
# 4. Remove "updrs_iii_percent" and "high_dose", so that df goes back to its original columns.

# Write your code here:

### 2.3.2 Data types

<div style="background-color:transparent; border-left:5px solid #48C9B0; padding:10px; margin-bottom:15px;">

 **Concept**

Every column has a **data type** (`dtype`) that tells pandas what kind of values it holds, and therefore what you can do with them. You can check them with `df.dtypes`.

| dtype | What it holds | Example in our data |
|:--|:--|:--|
| `int64` | Whole numbers | `clinic_id`, `ledd_mg` |
| `float64` | Decimal numbers (also used for any number column with missing values) | `updrs_iii` |
| `object` (shown as `str` in pandas 3) | Text | `patient_id`, `medication` |
| `bool` | `True` / `False` | `on_medication` (from 2.3.1) |
| `datetime64` | Dates and times | `visit_date`, once converted |

**Why it matters:** you cannot do arithmetic on numbers stored as text, or work with dates stored as text. When pandas reads a CSV file it **guesses** each type, and with messy EHR data it often guesses "text", for example because one cell says `"not recorded"` instead of a number.

| Conversion | Code |
|:--|:--|
| Text → numbers | `pd.to_numeric(df["column"], errors="coerce")` |
| Text → dates | `pd.to_datetime(df["column"])` |

- `errors="coerce"` means: *anything that cannot be turned into a number becomes `NaN`* (missing).
- Once a column contains dates, `.dt` gives you date tools such as `.dt.year`, `.dt.month` and `.dt.day`.

</div>

In [ ]:
#  Example 1: check the data types
df.dtypes

patient_id           str
sex                  str
date_of_birth        str
visit_date           str
clinic_id          int64
updrs_iii        float64
ledd_mg            int64
medication           str
bmi                  str
dtype: object

In [ ]:
#  Example 2: "bmi" was read as TEXT because some cells say "not recorded"
print("Type before:", df["bmi"].dtype)

# Convert it to numbers. errors="coerce" turns "not recorded" into NaN (missing)
df["bmi"] = pd.to_numeric(df["bmi"], errors="coerce")

print("Type after:", df["bmi"].dtype)
df[["patient_id", "bmi"]]

Type before: str
Type after: float64


,patient_id,bmi
0,P01,22.5
1,P02,25.0
2,P03,30.0
3,P04,28.0
4,P05,26.4
5,P01,22.7
6,P02,NaN
7,P03,31.2
8,P05,26.1
9,P06,21.9


In [ ]:
#  Example 3: dates are also read as text. pd.to_datetime() turns them into real dates...
df["visit_date"] = pd.to_datetime(df["visit_date"])

# ...which lets us use the .dt tools, for example to get the year of each visit
df["visit_year"] = df["visit_date"].dt.year

df[["patient_id", "visit_date", "visit_year"]]

,patient_id,visit_date,visit_year
0,P01,2023-01-10,2023
1,P02,2023-01-15,2023
2,P03,2023-02-01,2023
3,P04,2023-03-14,2023
4,P05,2023-04-21,2023
5,P01,2023-07-12,2023
6,P02,2023-09-03,2023
7,P03,2023-10-06,2023
8,P05,2023-11-30,2023
9,P06,2024-01-09,2024


In [ ]:
#  TASK 2.3.2
# 1. Convert the "date_of_birth" column to dates.
# 2. Create a new column "birth_year" with the year each patient was born.
# 3. Create a new column "age_at_visit" = visit_year - birth_year (the approximate age at each visit).
# 4. Display "patient_id", "date_of_birth", "birth_year" and "age_at_visit", then check df.dtypes.

# Write your code here:

### 2.3.3 Missing values

<div style="background-color:transparent; border-left:5px solid #F4D03F; padding:10px; margin-bottom:15px;">

 **Concept**

Missing values are everywhere in EHRs: a test was not done, a field was left empty, a measurement was not recorded. pandas marks them as **`NaN`** (*Not a Number*), or `NaT` (*Not a Time*) for dates.

| Code | What it does |
|:--|:--|
| `df.isna()` | `True` wherever a value is missing |
| `df.isna().sum()` | Number of missing values in each column (each `True` counts as 1) |
| `df[df["column"].isna()]` | The rows where `"column"` is missing (a filter, as in 2.2.3) |
| `df.dropna(subset=["column"])` | A new table **without** the rows where `"column"` is missing |
| `df["column"].fillna(value)` | Replace the missing values of `"column"` with `value` |

⚠️ Before removing or filling missing values, think about **why** they are missing. In our data, an empty medication field means *"no medication prescribed"* (those visits have `ledd_mg = 0`), so it can safely be filled in. A missing UPDRS III score means the assessment was not done, and we should not invent a value for it. In the next session you will learn more ways of filling in missing values in signals.

</div>

In [ ]:
#  Example 1: how many values are missing in each column?
df.isna().sum()

patient_id       0
sex              0
date_of_birth    0
visit_date       0
clinic_id        0
updrs_iii        2
ledd_mg          0
medication       2
bmi              2
visit_year       0
birth_year       0
age_at_visit     0
dtype: int64

In [ ]:
#  Example 2: which visits have no UPDRS III score?
df[df["updrs_iii"].isna()]

,patient_id,sex,date_of_birth,visit_date,clinic_id,updrs_iii,ledd_mg,medication,bmi,visit_year,birth_year,age_at_visit
6,P02,Female,1965-08-22,2023-09-03,102,NaN,300,rotigotine,NaN,2023,1965,58
13,P04,Male,1980-01-15,2024-06-04,103,NaN,0,NaN,27.6,2024,1980,44


In [ ]:
#  Example 3: remove those visits. We store the result in a NEW variable, so df keeps all its rows
df_with_updrs = df.dropna(subset=["updrs_iii"])
print("Rows in df:", len(df))
print("Rows after removing missing UPDRS III:", len(df_with_updrs))

Rows in df: 14
Rows after removing missing UPDRS III: 12


In [ ]:
#  Example 4: fill in missing values. An empty medication field means "no medication", so we write "none"
df["medication"] = df["medication"].fillna("none")
df[["patient_id", "ledd_mg", "medication"]]

,patient_id,ledd_mg,medication
0,P01,300,Co-careldopa
1,P02,150,Rotigotine
2,P03,800,Co-beneldopa
3,P04,0,none
4,P05,450,Pramipexole
5,P01,400,co-careldopa
6,P02,300,rotigotine
7,P03,850,co-beneldopa
8,P05,600,Co-careldopa + Entacapone
9,P06,400,Ropinirole


In [ ]:
#  TASK 2.3.3
# 1. How many BMI values are missing?
# 2. Show the visits where BMI is missing.
# 3. Create a new variable `df_with_bmi` without the visits where BMI is missing, and print how many rows it has.

# Write your code here:

### 2.3.4 Exploring categories

<div style="background-color:transparent; border-left:5px solid #85929E; padding:10px; margin-bottom:15px;">

 **Concept**

Some columns contain a small set of values that repeat, for example `sex`, `clinic_id` or `medication`. These are **categories**. Three tools help you explore them:

| Code | What it shows |
|:--|:--|
| `df["column"].unique()` | The different values that appear in the column |
| `df["column"].nunique()` | **How many** different values there are |
| `df["column"].value_counts()` | How many rows have each value |

💡 In the next session you will use `.value_counts()` to see how many recordings there are for each accelerometer task.

</div>

In [ ]:
#  Example: explore the "sex" column
print("Different values:", list(df["sex"].unique()))
print("Number of different values:", df["sex"].nunique())

df["sex"].value_counts()

Different values: ['Male', 'Female']
Number of different values: 2


sex
Male      8
Female    6
Name: count, dtype: int64

In [ ]:
#  TASK 2.3.4
# Explore the "medication" column:
# 1. How many different medication names are there?
# 2. How many visits are there for each medication name?
# Then answer the questions in the cell below.

# Write your code here:

##### What do you observe?

- How many **different drugs** are really in the table? Is it the same as the number pandas gives you?
- Why does pandas count `"Co-careldopa"`, `"co-careldopa "` and `"CO-CARELDOPA"` as three different values?
- What would go wrong if you counted the patients on each drug using this column as it is?

<br>

### 2.3.5 Cleaning text

<div style="background-color:transparent; border-left:5px solid #5DADE2; padding:10px; margin-bottom:15px;">

 **Concept**

EHR text is typed by people, so the same thing is often written in different ways: different capital letters, extra spaces, small variations. For pandas, `"Rotigotine"`, `"rotigotine"` and `" Rotigotine"` are **three different values**.

Adding `.str` after a text column gives you text tools that work on **every row at once**:

| Code | What it does |
|:--|:--|
| `.str.lower()` | Converts all letters to lowercase |
| `.str.upper()` | Converts all letters to uppercase |
| `.str.strip()` | Removes spaces at the start and at the end |
| `.str.replace("old", "new")` | Replaces a piece of text with another |
| `.str.contains("text")` | `True` if the value contains `"text"` |
| `.str.startswith("text")` | `True` if the value starts with `"text"` |

You can chain several tools one after the other: `df["column"].str.lower().str.strip()`.

💡 It is good practice to keep the original column and store the cleaned version in a **new** column, so you can always check what changed.

</div>

In [ ]:
#  Example 1: lowercase everything AND remove extra spaces, then store it in a new, clean column
df["medication_clean"] = df["medication"].str.lower().str.strip()
df[["medication", "medication_clean"]]

,medication,medication_clean
0,Co-careldopa,co-careldopa
1,Rotigotine,rotigotine
2,Co-beneldopa,co-beneldopa
3,none,none
4,Pramipexole,pramipexole
5,co-careldopa,co-careldopa
6,rotigotine,rotigotine
7,co-beneldopa,co-beneldopa
8,Co-careldopa + Entacapone,co-careldopa + entacapone
9,Ropinirole,ropinirole


In [ ]:
#  Example 2: now the counts make sense
df["medication_clean"].value_counts()

medication_clean
co-careldopa                 3
rotigotine                   3
co-beneldopa                 2
none                         2
co-careldopa + entacapone    2
pramipexole                  1
ropinirole                   1
Name: count, dtype: int64

In [ ]:
#  TASK 2.3.5
# Co-careldopa and co-beneldopa both contain levodopa, and their names contain the text "dopa".
# 1. Create a True/False column "levodopa_based" that is True when "medication_clean" contains "dopa".
# 2. Use this new column to show only the visits where the patient was taking a levodopa-based medication.
#    HINT: a True/False column is already a mask!

# Write your code here:

## 2.4 Grouping and combining tables

### 2.4.1 Grouping rows

<div style="background-color:transparent; border-left:5px solid #48C9B0; padding:10px; margin-bottom:15px;">

 **Concept**

In EHR data each patient has several rows (one per visit), but we often want **one answer per patient**, per clinic or per medication. `.groupby()` does this in three steps:

1. **Split** the table into groups, for example one group per `patient_id`;
2. **Apply** a calculation to each group, for example count its rows or find its largest value;
3. **Combine** the results into a new table with **one row per group**.

```python
df.groupby("column_to_group_by")["column_to_use"].calculation()
```

| Calculation | Result for each group |
|:--|:--|
| `.size()` | Number of rows (this one does not need a column: `df.groupby("column").size()`) |
| `.min()` | Smallest value (for dates: the earliest) |
| `.max()` | Largest value (for dates: the latest) |

</div>

In [ ]:
#  Example 1: how many visits does each patient have?
df.groupby("patient_id").size()

patient_id
P01    3
P02    3
P03    2
P04    2
P05    3
P06    1
dtype: int64

In [ ]:
#  Example 2: what is the highest dose (ledd_mg) each patient has been prescribed?
df.groupby("patient_id")["ledd_mg"].max()

patient_id
P01    500
P02    300
P03    850
P04      0
P05    750
P06    400
Name: ledd_mg, dtype: int64

In [ ]:
#  TASK 2.4.1
# 1. For each patient, find the date of their FIRST visit.
#    HINT: the first visit is the one with the earliest (smallest) date.
# 2. For each clinic ("clinic_id"), count how many visits took place there.

# Write your code here:

### 2.4.2 Combining tables

<div style="background-color:transparent; border-left:5px solid #F4D03F; padding:10px; margin-bottom:15px;">

 **Concept**

Clinical data is often split across several tables (visits, demographics, clinics...). There are two main ways of combining them.

**1. Stacking with `pd.concat()`**: puts tables with the **same columns** one under the other, for example to add new patients.

```python
pd.concat([table_1, table_2], ignore_index=True)
```

`ignore_index=True` renumbers the rows `0, 1, 2, ...` In the next session you will also use `pd.concat(..., axis=1)` to put tables side by side.

**2. Matching with `.merge()`**: adds the columns of another table by matching the values of a **shared column** (the *key*), for example `patient_id`.

```python
df.merge(other_table, on="key_column", how="left")
```

| `how=` | Rows kept |
|:--|:--|
| `"left"` | **All** the rows of the left table (`df`). Rows without a match get `NaN` in the new columns. *The most common choice.* |
| `"inner"` | Only the rows whose key appears in **both** tables |
| `"right"` | All the rows of the right table (`other_table`) |
| `"outer"` | All the rows of **both** tables |

![](images/IMG_2880.JPG)

</div>

In [ ]:
# recall the patients dataframe:

patients

,patient_id,age_at_diagnosis,smoking_status
0,P01,66,Former
1,P02,55,Never
2,P03,50,Never


In [ ]:
#  Example 1: stacking. Two new patients join the `patients` table from 2.1.1 (same columns)
new_patients = pd.DataFrame({
    "patient_id": ["P04", "P05"],
    "age_at_diagnosis": [43, 61],
    "smoking_status": ["Current", "Former"],
})

all_patients = pd.concat([patients, new_patients], ignore_index=True)
all_patients

,patient_id,age_at_diagnosis,smoking_status
0,P01,66,Former
1,P02,55,Never
2,P03,50,Never
3,P04,43,Current
4,P05,61,Former


In [ ]:
#  Example 2: matching. Add the information in `all_patients` to every visit in df, using "patient_id" as the key
df_merged = df.merge(all_patients, on="patient_id", how="left")
df_merged[["patient_id", "visit_date", "age_at_diagnosis", "smoking_status"]]

# Look at patient P06: they are not in `all_patients`, so their new columns are NaN.
# A left merge keeps every visit in df, even when there is no match.
# (age_at_diagnosis now shows decimals, e.g. 66.0, because a number column with NaN becomes float64, see 2.3.2)

,patient_id,visit_date,age_at_diagnosis,smoking_status
0,P01,2023-01-10,66.0,Former
1,P02,2023-01-15,55.0,Never
2,P03,2023-02-01,50.0,Never
3,P04,2023-03-14,43.0,Current
4,P05,2023-04-21,61.0,Former
5,P01,2023-07-12,66.0,Former
6,P02,2023-09-03,55.0,Never
7,P03,2023-10-06,50.0,Never
8,P05,2023-11-30,61.0,Former
9,P06,2024-01-09,NaN,NaN


In [ ]:
#  TASK 2.4.2
# The table below has information about each clinic (run this cell to create it):
clinics = pd.DataFrame({
    "clinic_id": [101, 102, 103],
    "clinic_name": ["St Mary's", "Queen Elizabeth", "Royal Infirmary"],
    "city": ["London", "Birmingham", "Manchester"],
})

# 1. Merge `clinics` into `df_merged` (from the example above), using "clinic_id" as the key.
#    Keep all the visits (a left merge) and store the result in a variable called `df_full`.
# 2. Display the "patient_id", "visit_date", "clinic_name" and "city" columns of df_full.

# Write your code here:

## 2.5 Transforming and summarising data: map, apply, agg, and transform

### What is a transformation?

<div style="background-color:transparent; border-left:5px solid #85929E; padding:10px; margin-bottom:15px;">

So far you have changed columns with simple arithmetic (`df["ledd_mg"] / 1000`) and with text tools (`.str.lower()`). Often, though, you need **your own rules**, for example *"turn each UPDRS III score into mild, moderate or severe"*, or *"for each visit, how many days have passed since that patient's first visit?"*. pandas has four tools for this. They differ in **what they work on** and in **how many rows come out**:

| Tool | Works on | What it does | Rows that come out |
|:--|:--|:--|:--|
| `.map()` | One column | Replaces each value using a **dictionary** | Same number as `df` |
| `.apply()` | One column, or whole rows | Runs **your own function** on each value (or on each row) | Same number as `df` |
| `.groupby().transform()` | Groups | Calculates something **per group** and copies the result back to **every row** of that group | Same number as `df` |
| `.groupby().agg()` | Groups | Calculates one or more things **per group** and **summarises** each group into a single row | One per group |

The first three are **transformations**: every row gets its own new value, so the result can be stored as a **new column** of `df`. The last one is an **aggregation**: many rows are summarised into fewer rows, so the result is a **new, smaller table**.

Here is the difference for patient P01, who has 3 visits:

```
ledd_mg of P01       .agg("max")          .transform("max")
     300                                        500
     400      ──►        500        or          500
     500                                        500
   3 rows               1 row                 3 rows
```

`.transform()` and `.agg()` always come after a `.groupby()`, which you met in 2.4.1.

</div>

### 2.5.1 Recoding values with `.map()`

<div style="background-color:transparent; border-left:5px solid #5DADE2; padding:10px; margin-bottom:15px;">

 **Concept**

`.map()` goes through a column and replaces each value with a new one. Its most common use is **recoding** with a **dictionary** that pairs each old value with its new value:

```python
df["new_column"] = df["column"].map({"old_value_1": "new_value_1", "old_value_2": "new_value_2"})
```

Typical uses in EHR data: shortening labels (`"Male"` → `"M"`), turning answers into numbers for a model (`"Yes"` → `1`), or grouping several values into a broader category (drug name → drug class).

⚠️ Any value that is **not** in the dictionary becomes `NaN`, and pandas does **not** warn you. Always check the result, for example with `.isna().sum()` or `.value_counts()`.

</div>

In [ ]:
#  Example 1: shorten the "sex" labels. The dictionary means {"old value": "new value"}
sex_codes = {"Male": "M", "Female": "F"}

df["sex_short"] = df["sex"].map(sex_codes)
df[["patient_id", "sex", "sex_short"]].head()

,patient_id,sex,sex_short
0,P01,Male,M
1,P02,Female,F
2,P03,Female,F
3,P04,Male,M
4,P05,Male,M


In [ ]:
#  Example 2: what happens if a value is missing from the dictionary? Here we forgot "Female"...
df["sex"].map({"Male": "M"}).head()

# ...so every "Female" became NaN, without any warning. Always check your result!

0      M
1    NaN
2    NaN
3      M
4      M
Name: sex, dtype: str

In [ ]:
#  TASK 2.5.1
# The dictionary below groups the cleaned medication names (from 2.3.5) into drug classes:
drug_classes = {
    "co-careldopa": "Levodopa",
    "co-beneldopa": "Levodopa",
    "co-careldopa + entacapone": "Levodopa + COMT inhibitor",
    "rotigotine": "Dopamine agonist",
    "pramipexole": "Dopamine agonist",
    "none": "No medication",
}

# 1. Use .map() to create a new column "drug_class" from the "medication_clean" column.
# 2. Check your result with .value_counts() and .isna().sum(). Did every visit get a drug class?
# 3. One medication is missing from the dictionary. Find which one (HINT: filter the rows where "drug_class" is missing, 2.3.3),
#    add it to `drug_classes` (it is a dopamine agonist) and create the "drug_class" column again.
#    HINT: you can add an entry to a dictionary with  my_dictionary["new key"] = "new value"
#          (or simply edit the dictionary above and run the cell again)

# Write your code here:

### 2.5.2 Your own rules with `.apply()` and `lambda`

<div style="background-color:transparent; border-left:5px solid #48C9B0; padding:10px; margin-bottom:15px;">

 **Concept**

In the last session you wrote your own functions with `def`. `.apply()` runs **your own function** on every value of a column:

```python
def my_function(value):
    # ... your rules, for example with if / elif / else ...
    return new_value

df["new_column"] = df["column"].apply(my_function)   # no brackets after my_function!
```

pandas calls `my_function` once for every row, giving it that row's value, and collects everything it returns into a new column. Use `.apply()` when the rule is too complex for a dictionary.

**`lambda`: a function in one line.** For very short rules you can write the function directly inside `.apply()`. These two are the same:

```python
lambda dose: dose / 1000          # input name, a colon, then what to return

def to_grams(dose):
    return dose / 1000
```

**Using several columns: `axis=1`.** `df.apply(my_function, axis=1)` runs the function on **each row** of the whole table. The function receives the complete row, and reads any column from it with `row["column_name"]`.

⚠️ Inside your function, deal with missing values **first** (`if pd.isna(value): ...`). Any comparison with `NaN`, such as `NaN <= 32`, gives `False`, so a missing value would silently end up in the wrong category.

💡 If pandas already has a simple way to do something (`df["ledd_mg"] / 1000`, `.str.lower()`), use it: it is faster and easier to read. Keep `.apply()` for your own rules. You will use `.apply()` with `lambda` again in the next session.

</div>

In [ ]:
#  Example 1: classify each MDS-UPDRS Part III score as mild, moderate or severe
# (cut-offs proposed by Martínez-Martín et al., 2015: mild 0-32, moderate 33-58, severe 59 or more)
def updrs_severity(score):
    if pd.isna(score):          # deal with missing values first!
        return "not assessed"
    elif score <= 32:
        return "mild"
    elif score <= 58:
        return "moderate"
    else:
        return "severe"

df["motor_severity"] = df["updrs_iii"].apply(updrs_severity)
df[["patient_id", "updrs_iii", "motor_severity"]]

,patient_id,updrs_iii,motor_severity
0,P01,22.0,mild
1,P02,30.0,mild
2,P03,40.0,moderate
3,P04,12.0,mild
4,P05,35.0,moderate
5,P01,25.0,mild
6,P02,NaN,not assessed
7,P03,42.0,moderate
8,P05,38.0,moderate
9,P06,27.0,mild


In [ ]:
#  Example 2: a short rule written as a lambda.
# "A if condition else B" gives A when the condition is True, and B when it is False
df["treatment_status"] = df["ledd_mg"].apply(lambda dose: "treated" if dose > 0 else "untreated")
df[["patient_id", "ledd_mg", "treatment_status"]]

,patient_id,ledd_mg,treatment_status
0,P01,300,treated
1,P02,150,treated
2,P03,800,treated
3,P04,0,untreated
4,P05,450,treated
5,P01,400,treated
6,P02,300,treated
7,P03,850,treated
8,P05,600,treated
9,P06,400,treated


In [ ]:
#  Example 3: a function that uses SEVERAL columns of each row (axis=1): a data-quality check of each visit
def record_check(row):
    if pd.isna(row["updrs_iii"]) and pd.isna(row["bmi"]):
        return "missing UPDRS III and BMI"
    elif pd.isna(row["updrs_iii"]):
        return "missing UPDRS III"
    elif pd.isna(row["bmi"]):
        return "missing BMI"
    else:
        return "complete"

df["record_status"] = df.apply(record_check, axis=1)
df[["patient_id", "visit_date", "updrs_iii", "bmi", "record_status"]]

,patient_id,visit_date,updrs_iii,bmi,record_status
0,P01,2023-01-10,22.0,22.5,complete
1,P02,2023-01-15,30.0,25.0,complete
2,P03,2023-02-01,40.0,30.0,complete
3,P04,2023-03-14,12.0,28.0,complete
4,P05,2023-04-21,35.0,26.4,complete
5,P01,2023-07-12,25.0,22.7,complete
6,P02,2023-09-03,NaN,NaN,missing UPDRS III and BMI
7,P03,2023-10-06,42.0,31.2,complete
8,P05,2023-11-30,38.0,26.1,complete
9,P06,2024-01-09,27.0,21.9,complete


In [ ]:
#  TASK 2.5.2
# 1. Write a function called bmi_category that takes ONE BMI value and returns:
#       "not recorded"   if the value is missing
#       "underweight"    if BMI is below 18.5
#       "healthy"        if BMI is 18.5 or more, but below 25
#       "overweight"     if BMI is 25 or more, but below 30
#       "obese"          if BMI is 30 or more
#    Then use .apply() to create a new column "bmi_category" from the "bmi" column.
# 2. Use .apply() with a lambda to create a column "dose_level" that is "high" when "ledd_mg" is above 500,
#    and "standard" otherwise.

# Write your code here:

### 2.5.3 Summarising groups with `.agg()`

<div style="background-color:transparent; border-left:5px solid #F4D03F; padding:10px; margin-bottom:15px;">

 **Concept**

In 2.4.1 you used `.groupby()` with **one** calculation, such as `.max()`. `.agg()` (short for *aggregate*) lets you ask for **several calculations at once** and build a **summary table** with one row per group.

**Several calculations on one column:**

```python
df.groupby("patient_id")["ledd_mg"].agg(["min", "max"])
```

**Different calculations on different columns, with column names you choose** (*named aggregation*):

```python
summary = df.groupby("patient_id").agg(
    new_column_1=("column_to_use", "calculation"),
    new_column_2=("other_column", "other_calculation"),
)
```

Read each line as: *new column name = (which column, which calculation)*.

| Calculation | Result for each group |
|:--|:--|
| `"count"` | Number of non-missing values |
| `"nunique"` | Number of **different** values |
| `"min"`, `"max"` | Smallest / largest value (earliest / latest for dates) |
| `"sum"` | Total |

After a `.groupby()`, the group names (e.g. `patient_id`) become the **index** of the result. Add `.reset_index()` to turn them back into a normal column, which you need if you want to merge the summary with another table (2.4.2).

💡 You can also group by more than one column with a list: `df.groupby(["clinic_id", "sex"])`.

</div>

In [ ]:
#  Example 1: the lowest and the highest dose of each patient, in one table
df.groupby("patient_id")["ledd_mg"].agg(["min", "max"])

,min,max
patient_id,,
P01,300,500
P02,150,300
P03,800,850
P04,0,0
P05,450,750
P06,400,400


In [ ]:
#  Example 2: a patient-level summary table (one row per patient), with column names we choose
patient_summary = df.groupby("patient_id").agg(
    n_visits=("visit_date", "count"),
    first_visit=("visit_date", "min"),
    last_visit=("visit_date", "max"),
    highest_ledd=("ledd_mg", "max"),
).reset_index()

patient_summary

,patient_id,n_visits,first_visit,last_visit,highest_ledd
0,P01,3,2023-01-10,2024-02-08,500
1,P02,3,2023-01-15,2024-03-20,300
2,P03,2,2023-02-01,2023-10-06,850
3,P04,2,2023-03-14,2024-06-04,0
4,P05,3,2023-04-21,2024-05-17,750
5,P06,1,2024-01-09,2024-01-09,400


In [ ]:
#  TASK 2.5.3
# 1. Create a table called `clinic_summary` with ONE ROW PER CLINIC ("clinic_id") and these columns:
#       n_visits        number of visits at that clinic                       HINT: count the "visit_date" values
#       n_patients      number of DIFFERENT patients seen at that clinic      HINT: "nunique"
#       highest_updrs   highest UPDRS III score recorded at that clinic
#    Use .reset_index() so that "clinic_id" is a normal column.
# 2. Merge the `clinics` table (from Task 2.4.2) into `clinic_summary`, so that each row also shows the clinic's name and city.

# Write your code here:

### 2.5.4 Group transformations with `.transform()`

<div style="background-color:transparent; border-left:5px solid #85929E; padding:10px; margin-bottom:15px;">

 **Concept**

`.agg()` shrinks each group to **one row**. Often, though, you want to keep **every visit** and add information about the patient it belongs to, for example *"how many days after this patient's first visit did this visit happen?"*. That is what `.transform()` does: it calculates something per group and copies the result back to **every row** of that group. The result has the same number of rows as `df`, so you can store it as a new column:

```python
df["new_column"] = df.groupby("patient_id")["column"].transform("calculation")
```

The same calculations as in `.agg()` work here (`"min"`, `"max"`, `"count"`, ...). You can also give `.transform()` a `lambda`: it then receives **all the values of one group at once**, for example all the doses of one patient: `lambda doses: doses - doses.min()`.

This is how you compare each observation with the patient's **own** values, which is very common in longitudinal EHR analyses.

💡 Subtracting one date from another gives a time difference; `.dt.days` turns it into a number of days.

</div>

In [ ]:
#  Example 1: the date of each patient's first visit, copied to every one of their visits
df["first_visit"] = df.groupby("patient_id")["visit_date"].transform("min")

# Now each visit can be compared with the patient's own first visit
df["days_since_first_visit"] = (df["visit_date"] - df["first_visit"]).dt.days

df[["patient_id", "visit_date", "first_visit", "days_since_first_visit"]].sort_values(["patient_id", "visit_date"])

,patient_id,visit_date,first_visit,days_since_first_visit
0,P01,2023-01-10,2023-01-10,0
5,P01,2023-07-12,2023-01-10,183
10,P01,2024-02-08,2023-01-10,394
1,P02,2023-01-15,2023-01-15,0
6,P02,2023-09-03,2023-01-15,231
11,P02,2024-03-20,2023-01-15,430
2,P03,2023-02-01,2023-02-01,0
7,P03,2023-10-06,2023-02-01,247
3,P04,2023-03-14,2023-03-14,0
13,P04,2024-06-04,2023-03-14,448


In [ ]:
#  Example 2: with a lambda. How much higher is each dose than the patient's own lowest dose?
# The lambda receives all the doses of ONE patient at a time
df["ledd_above_lowest"] = df.groupby("patient_id")["ledd_mg"].transform(lambda doses: doses - doses.min())

df[["patient_id", "visit_date", "ledd_mg", "ledd_above_lowest"]].sort_values(["patient_id", "visit_date"])

,patient_id,visit_date,ledd_mg,ledd_above_lowest
0,P01,2023-01-10,300,0
5,P01,2023-07-12,400,100
10,P01,2024-02-08,500,200
1,P02,2023-01-15,150,0
6,P02,2023-09-03,300,150
11,P02,2024-03-20,300,150
2,P03,2023-02-01,800,0
7,P03,2023-10-06,850,50
3,P04,2023-03-14,0,0
13,P04,2024-06-04,0,0


In [ ]:
#  TASK 2.5.4
# 1. Add a column "n_visits" with the number of visits of each patient, repeated on every one of their visits.
#    HINT: count the "visit_date" values per patient with .transform()
# 2. Many longitudinal analyses need at least 2 visits per patient. Use your new column to create `df_followup`,
#    with only the visits of patients who have 2 or more visits. Which patient is removed?

# Write your code here:

### 2.5.5 Choosing the right tool

<div style="background-color:transparent; border-left:5px solid #5DADE2; padding:10px; margin-bottom:15px;">

 **Concept**

When you are not sure which tool to use, ask yourself two questions:

**1. Do I want one result per row, or one row per group?**
- One row per group (a summary table) → `.groupby().agg()`
- One result per row (a new column) → go to question 2

**2. What does each new value depend on?**
- Only on that row's value, through a fixed lookup → `.map(dictionary)`
- On that row's value, through a rule I write → `.apply(function)` (or `.apply(function, axis=1)` to use several columns of the row)
- On the **other rows of the same group** (e.g. the same patient's other visits) → `.groupby().transform()`

| Tool | Works on | Rows that come out | Typical EHR use | Example from this section |
|:--|:--|:--|:--|:--|
| `.map(dictionary)` | One column | Same | Recode labels or group values into categories | `df["sex"].map({"Male": "M", "Female": "F"})` |
| `.apply(function)` | One column | Same | Your own rule for each value | `df["updrs_iii"].apply(updrs_severity)` |
| `.apply(function, axis=1)` | Rows | Same | A rule that uses several columns | `df.apply(record_check, axis=1)` |
| `.groupby().transform()` | Groups | Same | Compare each visit with the patient's own values | `df.groupby("patient_id")["visit_date"].transform("min")` |
| `.groupby().agg()` | Groups | One per group | Summary tables | `df.groupby("patient_id").agg(n_visits=("visit_date", "count"))` |

</div>

In [ ]:
#  Example: the same calculation with .agg() and with .transform(). Compare the number of rows!
highest_agg = df.groupby("patient_id")["ledd_mg"].agg("max")
highest_transform = df.groupby("patient_id")["ledd_mg"].transform("max")

print("df has", len(df), "rows (one per visit)")
print(".agg('max') gives", len(highest_agg), "rows (one per patient)")
print(".transform('max') gives", len(highest_transform), "rows (one per visit), so it can become a new column of df")

df has 14 rows (one per visit)
.agg('max') gives 6 rows (one per patient)
.transform('max') gives 14 rows (one per visit), so it can become a new column of df


In [ ]:
#  TASK 2.5.5
# For each question, first decide which tool you need (.map, .apply, .transform or .agg), then write the code.
# 1. Create a column "clinic_city" with the city of the clinic of each visit, using this dictionary:
#       {101: "London", 102: "Birmingham", 103: "Manchester"}
#    (you added the same information with .merge() in Task 2.4.2; for a single column, .map() is a quick alternative)
# 2. Create a table with ONE ROW PER PATIENT showing their lowest and highest UPDRS III score.
# 3. Add a column "highest_updrs_patient" with each patient's highest UPDRS III score, repeated on every visit.
#    Then create a True/False column "is_worst_visit" that is True on the visit(s) where the patient had their highest score.

# Write your code here:

## 2.6 Putting it all together

You now know enough pandas to turn a raw EHR export into a clean table ready for analysis. This final task uses steps from the whole session; the section where each step was introduced is shown in brackets.

In [ ]:
#  FINAL TASK
# Start again from the raw file and build a clean table of the visits where patients were on medication.
#
# 1. Load "ehr_parkinsons.csv" into a new variable called `raw`.                                  (2.1.2)
# 2. Convert "bmi" to numbers and "visit_date" to dates.                                          (2.3.2)
# 3. Fill the missing medication names with "none", then create a "medication_clean" column
#    in lowercase and without extra spaces.                                                       (2.3.3, 2.3.5)
# 4. Keep only the visits where the patient was taking medication (ledd_mg above 0),
#    and then remove the visits without a UPDRS III score. Call the result `clean`.               (2.2.3, 2.3.3)
# 5. Keep only these columns: "patient_id", "visit_date", "updrs_iii", "ledd_mg",
#    "medication_clean" and "bmi".                                                                (2.2.1)
# 6. Sort the visits by patient, and then by visit date.                                          (2.2.5)
# 7. Count how many visits each patient has in your clean table.                                  (2.4.1)
# 8. Save the clean table as "ehr_parkinsons_clean.csv" (without the index).                      (2.1.2)

# Write your code here:

## Summary: today's pandas toolbox

| Topic | Code | What it does |
|:--|:--|:--|
| Create | `pd.DataFrame({"col": [...]})` | Build a table from a dictionary |
| Load / save | `pd.read_csv("file.csv")`, `df.to_csv("file.csv", index=False)` | Read and write CSV files |
| Inspect | `df.head()`, `df.tail()`, `df.shape`, `df.columns`, `df.info()` | Look at the table |
| Select columns | `df["col"]`, `df[["col_1", "col_2"]]` | One column (Series) or several (DataFrame) |
| Select rows | `df.iloc[0:3]`, `df.loc[4, "col"]` | By position or by label |
| Filter | `df[df["col"] > 30]`, `df[(cond_1) & (cond_2)]` | Keep the rows that meet one or more conditions |
| Sort | `df.sort_values("col", ascending=False)` | Reorder the rows |
| Change columns | `df["new"] = ...`, `df.rename(columns={...})`, `df.drop(columns=[...])` | Add, rename and remove columns |
| Data types | `df.dtypes`, `pd.to_numeric()`, `pd.to_datetime()`, `.dt.year` | Check and fix types |
| Missing values | `.isna().sum()`, `.dropna(subset=[...])`, `.fillna(value)` | Find, remove or fill in missing values |
| Categories | `.unique()`, `.nunique()`, `.value_counts()` | Explore repeated values |
| Text | `.str.lower()`, `.str.strip()`, `.str.contains()` | Clean and search text |
| Group | `df.groupby("col")["other_col"].max()`, `.size()` | One result per group |
| Combine | `pd.concat([df_1, df_2])`, `df.merge(other, on="key", how="left")` | Stack or match tables |
| Recode values | `df["col"].map({"old": "new"})` | Replace each value using a dictionary |
| Your own rules | `df["col"].apply(my_function)`, `df.apply(my_function, axis=1)`, `lambda x: ...` | Run your own function on each value or on each row |
| Group summary | `df.groupby("col").agg(new_name=("other_col", "max"))` | One row per group, several calculations at once |
| Group transformation | `df.groupby("col")["other_col"].transform("min")` | One value per row, calculated within its group |



> ⚠️ Remember: most pandas methods give you back a **new** table. Store the result in a variable (`df = df.drop(...)`) if you want to keep the change.

---

#### **Next session: time-series data as biological signals**

You will load real accelerometer recordings with `pd.read_csv()`, explore them with `.head()`, `.shape` and `.value_counts()`, select one subject, wrist and task by combining conditions with `&`, add, rename and combine columns, and use `.apply()` with `lambda` to filter each accelerometer column, using everything you practised today.